# Ensemble-POS-Vorhersage auf fester Tokenisierung


## Schritt 0 — Konfiguration


In [1]:
from pathlib import Path

GOLD_PFAD    = (Path("..") / "02_quelldaten"
                / "goldstandard_v1.conll")     # nur Quelle der Tokenisierung
AUSGABE_PFAD = (Path("..") / "05_ergebnisse" / "pos"
                / "ensemble_pred.conll")       # Systemvorhersage, NICHT Gold

BACKENDS     = ["spacy", "stanza", "flair"]    # Teilnehmer der Abstimmung
SPACY_MODELL = "de_core_news_lg"               # wie im Original-Ensemble
PRIORITAET   = "spacy"                         # Gewinner bei Gleichstand

print(f"Gold: {GOLD_PFAD} -> Ausgabe: {AUSGABE_PFAD}")
print(f"Backends: {BACKENDS} | spaCy-Modell: {SPACY_MODELL} | "
      f"Gleichstand -> {PRIORITAET}")

Gold: ../02_quelldaten/goldstandard_v1.conll -> Ausgabe: ../05_ergebnisse/pos/ensemble_pred.conll
Backends: ['spacy', 'stanza', 'flair'] | spaCy-Modell: de_core_news_lg | Gleichstand -> spacy


## Schritt 1 — Golddatei strukturerhaltend einlesen



In [ ]:
def lese_struktur(pfad):
    zeilen, saetze, cur = [], [], []
    for nr, roh in enumerate(open(pfad, encoding="utf-8"), 1):
        z = roh.rstrip("\n")
        if z.startswith("#"):
            zeilen.append(("kommentar", z))
        elif not z.strip():
            zeilen.append(("leer", ""))
            if cur:
                saetze.append(cur)
                cur = []
        else:
            p = z.split("\t")
            if len(p) != 3:
                raise ValueError(f"{pfad}:{nr}: erwarte 3 Spalten: {z!r}")
            zeilen.append(("token", p[0]))
            cur.append(p[0])
    if cur:
        saetze.append(cur)
    return zeilen, saetze

zeilen, saetze = lese_struktur(GOLD_PFAD)
n_tok = sum(len(s) for s in saetze)
print(f"Eingelesen: {len(saetze)} Sätze | {n_tok} Tokens | "
      f"{sum(1 for t, _ in zeilen if t == 'kommentar')} Kommentarzeilen")

Eingelesen: 150 Sätze | 4137 Tokens | 202 Kommentarzeilen


## Schritt 2 — STTS→UPOS-Übersetzungstabelle


In [3]:
STTS_TO_UPOS = {
    "NN": "NOUN", "NE": "PROPN",
    "VVFIN": "VERB", "VVINF": "VERB", "VVPP": "VERB", "VVIMP": "VERB",
    "VAFIN": "AUX", "VAINF": "AUX", "VAPP": "AUX",
    "VMFIN": "AUX", "VMINF": "AUX", "VMPP": "AUX",
    "ADJA": "ADJ", "ADJD": "ADJ",
    "ART": "DET", "PDAT": "DET", "PIAT": "DET", "PPOSAT": "DET",
    "PPER": "PRON", "PRF": "PRON", "PPOSS": "PRON", "PRELS": "PRON",
    "PWS": "PRON", "PWAV": "ADV", "PWAT": "DET",
    "ADV": "ADV", "PROAV": "ADV", "PAV": "ADV",
    "APPR": "ADP", "APPRART": "ADP", "APPO": "ADP", "APZR": "ADP",
    "KON": "CCONJ", "KOUS": "SCONJ", "KOUI": "SCONJ", "KOKOM": "CCONJ",
    "CARD": "NUM", "ITJ": "INTJ",
    "PTKNEG": "PART", "PTKZU": "PART", "PTKVZ": "PART", "PTKANT": "PART",
    "PTKA": "PART",
    "PDS": "PRON", "PIS": "PRON", "PIDAT": "DET", "PRELAT": "DET",
    "XY": "X", "FM": "X", "TRUNC": "X",
    "$.": "PUNCT", "$,": "PUNCT", "$(": "PUNCT",
}
print(f"{len(STTS_TO_UPOS)} STTS-Tags abgedeckt "
      f"(5 gegenüber dem Original ergänzt).")

53 STTS-Tags abgedeckt (5 gegenüber dem Original ergänzt).


## Schritt 3 — Die drei Backends auf fester Tokenisierung


In [4]:
def mach_spacy(modell):
    import spacy
    from spacy.tokens import Doc
    nlp = spacy.load(modell)
    schwer = [n for n in ("parser", "lemmatizer", "attribute_ruler", "ner")
              if n in nlp.pipe_names]
    def tagge(tokens):
        with nlp.select_pipes(disable=schwer):
            doc = Doc(nlp.vocab, words=tokens)
            for _, proc in nlp.pipeline:
                doc = proc(doc)
        return [t.pos_ or t.tag_ for t in doc]
    return tagge


def mach_stanza():
    import stanza
    # Einmalig nötig, falls noch nicht geschehen: stanza.download("de")
    nlp = stanza.Pipeline("de", processors="tokenize,pos",
                          tokenize_pretokenized=True, verbose=False)
    def tagge(tokens):
        doc = nlp([tokens])                      # 1 Satz, fest tokenisiert
        tags = [w.upos for s in doc.sentences for w in s.words]
        if len(tags) != len(tokens):
            raise RuntimeError("Stanza-Ausgabe nicht tokengleich — "
                               "tokenize_pretokenized aktiv?")
        return tags
    return tagge


def mach_flair():
    from flair.models import SequenceTagger
    from flair.data import Sentence
    tagger = SequenceTagger.load("de-pos")
    def tagge(tokens):
        sent = Sentence(" ".join(tokens), use_tokenizer=False)
        tagger.predict(sent)
        if len(sent) != len(tokens):
            raise RuntimeError("Flair-Ausgabe nicht tokengleich.")
        return [STTS_TO_UPOS.get(t.get_label("pos").value,
                                 t.get_label("pos").value) for t in sent]
    return tagge

print("Backend-Fabriken definiert.")

Backend-Fabriken definiert.


## Schritt 4 — Backends laden


In [5]:
tagger = {}
for name in BACKENDS:
    try:
        if name == "spacy":
            tagger[name] = mach_spacy(SPACY_MODELL)
        elif name == "stanza":
            tagger[name] = mach_stanza()
        elif name == "flair":
            tagger[name] = mach_flair()
        else:
            raise ValueError(f"unbekanntes Backend: {name}")
        print(f"  geladen: {name}")
    except Exception as e:
        print(f"  WARNUNG: '{name}' nicht geladen "
              f"({type(e).__name__}: {e}) — wird übersprungen.")

if not tagger:
    raise RuntimeError("Kein Backend verfügbar — Umgebung prüfen "
                       "(3.13er-venv mit spacy/stanza/flair).")
if len(tagger) < len(BACKENDS):
    print(f"HINWEIS: nur {len(tagger)}/{len(BACKENDS)} Backends aktiv — "
          f"Mehrheitssemantik weicht vom Drei-Stimmen-Original ab.")

/home/timc/Workspace/fairytale-personality-profiling/venv_fairytale-profiling/lib/python3.13/site-packages/torch/cuda/__init__.py:180: UserWarning: CUDA initialization: The NVIDIA driver on your system is too old (found version 12090). Please update your GPU driver by downloading and installing a new version from the URL: http://www.nvidia.com/Download/index.aspx Alternatively, go to: https://pytorch.org to install a PyTorch version that has been compiled with your version of the CUDA driver. (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:119.)
  return torch._C._cuda_getDeviceCount() > 0


  geladen: spacy


/home/timc/Workspace/fairytale-personality-profiling/venv_fairytale-profiling/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


  geladen: stanza
2026-09-13 17:54:08,740 SequenceTagger predicts: Dictionary with 58 tags: <unk>, O, APPR, ART, ADJA, NN, VVFIN, PIS, NE, FM, $,, KON, $., CARD, APPRART, $(, PROAV, KOUS, PPER, ADV, VVINF, VAFIN, VMFIN, ADJD, PTKVZ, PTKNEG, KOKOM, PIDAT, PIAT, VVPP, PRF, PTKA, TRUNC, PPOSAT, VVIZU, PTKZU, VAINF, VMINF, PWAV, PDAT, PRELS, KOUI, APPO, VAPP, PWAT, PWS, VVIMP, APZR, PDS, PRELAT
  geladen: flair


## Schritt 5 — Mehrheitsentscheid und Ensemble-Kern


In [ ]:
from collections import Counter

def mehrheit(stimmen, prioritaet):
    haeufig = Counter(stimmen.values()).most_common()
    if len(haeufig) == 1 or haeufig[0][1] > haeufig[1][1]:
        return haeufig[0][0]
    return stimmen.get(prioritaet, haeufig[0][0])


def ensemble_auf_saetzen(saetze, tagger, prioritaet):
    ergebnis, stat, dissens = [], Counter(), Counter()
    for tokens in saetze:
        pro_backend = {name: fn(tokens) for name, fn in tagger.items()}
        for i in range(len(tokens)):
            stimmen = {n: tags[i] for n, tags in pro_backend.items()}
            ergebnis.append(mehrheit(stimmen, prioritaet))
            arten = len(set(stimmen.values()))
            if arten == 1:
                stat["einstimmig"] += 1
            elif Counter(stimmen.values()).most_common(1)[0][1] > 1:
                stat["mehrheit_2zu1"] += 1
                dissens[tuple(sorted(set(stimmen.values())))] += 1
            else:
                stat["gleichstand"] += 1
                dissens[tuple(sorted(set(stimmen.values())))] += 1
    return ergebnis, stat, dissens

print("Ensemble-Kern definiert.")

Ensemble-Kern definiert.


## Schritt 6 — Abstimmung ausführen


In [7]:
pos_final, stat, dissens = ensemble_auf_saetzen(saetze, tagger, PRIORITAET)

gesamt = sum(stat.values())
print(f"Abstimmung über {gesamt} Tokens:")
for k in ("einstimmig", "mehrheit_2zu1", "gleichstand"):
    print(f"  {k:<14} {stat.get(k, 0):>5}  "
          f"({100 * stat.get(k, 0) / gesamt:.1f} %)")
if dissens:
    print("Häufigste Dissens-Kombinationen:")
    for kombi, n in dissens.most_common(8):
        print(f"  {'/'.join(kombi):<20} {n:>4}x")

Abstimmung über 4137 Tokens:
  einstimmig      3717  (89.8 %)
  mehrheit_2zu1    386  (9.3 %)
  gleichstand       34  (0.8 %)
Häufigste Dissens-Kombinationen:
  ADJ/ADV                86x
  DET/PRON               46x
  ADV/PART               30x
  ADP/PART               26x
  AUX/VERB               26x
  NOUN/PROPN             25x
  ADP/DET                21x
  ADV/SCONJ              20x


## Schritt 7 — Ausgabe schreiben und validieren


In [8]:
def schreibe_conll(pfad, zeilen, pos_pro_token):
    it = iter(pos_pro_token)
    aus = []
    for typ, inhalt in zeilen:
        if typ == "token":
            aus.append(f"{inhalt}\t{next(it)}\tO")
        else:
            aus.append(inhalt)
    with open(pfad, "w", encoding="utf-8") as f:
        f.write("\n".join(aus) + "\n")

schreibe_conll(AUSGABE_PFAD, zeilen, pos_final)

kontrolle_zeilen, kontrolle_saetze = lese_struktur(AUSGABE_PFAD)
assert [t for s in kontrolle_saetze for t in s] == \
       [t for s in saetze for t in s], "Tokenfolge verändert!"
assert len(kontrolle_zeilen) == len(zeilen), "Zeilenstruktur verändert!"
print(f"Geschrieben und validiert: {AUSGABE_PFAD} "
      f"({len(kontrolle_saetze)} Sätze, tokengleich).")

print("\nNächster Schritt — Auswertung gegen das blind erhobene POS-Gold in :", AUSGABE_PFAD)

Geschrieben und validiert: ../05_ergebnisse/pos/ensemble_pred.conll (150 Sätze, tokengleich).

Nächster Schritt — Auswertung gegen das blind erhobene POS-Gold in: ../05_ergebnisse/pos/ensemble_pred.conll